# Phase 2 — High-Level Dataset Evaluation

**Self-contained** notebook that compares several manually-downloaded datasets **at a high level** and presents a side-by-side matrix, so you can pick a dataset by judgement.

## Design
- **Input:** manually downloaded + extracted CSVs placed in **`src/eda/input/`**.
- **Presentation only.** This notebook does *not* auto-select. It shows cheap comparative stats; the choice is yours (validated deeper in Phase 3).
- **Fast.** Sampled reads (`nrows=`) keep runtime short even for large archives.
- No imports of project `.py` modules; detectors + ticker extraction are inline.

## Datasets under comparison
The following manually-downloaded Kaggle datasets are evaluated here (placed under `src/eda/input/<folder>/`):

1. **`leukipp_pennystocks/submissions_reddit.csv`** — r/pennystocks submissions, from [leukipp/reddit-finance-data](https://www.kaggle.com/datasets/leukipp/reddit-finance-data).
2. **`leukipp_wallstreetbets/submissions_reddit.csv`** — r/wallstreetbets submissions, from the same [leukipp/reddit-finance-data](https://www.kaggle.com/datasets/leukipp/reddit-finance-data).
3. **`davidwallach_financial-tweets/stockerbot-export.csv`** — financial tweets (StockerBot), from [davidwallach/financial-tweets](https://www.kaggle.com/datasets/davidwallach/financial-tweets/data).
4. **`vivekrathi055_sentiment-analysis-on-financial-tweets/stockerbot-export1.csv`** — from [vivekrathi055/sentiment-analysis-on-financial-tweets](https://www.kaggle.com/datasets/vivekrathi055/sentiment-analysis-on-financial-tweets/data). This is a mirror of the same StockerBot export as (3), so their profiled stats are effectively identical.

**Output:** `src/eda/output/highlevel_comparison.csv`.

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# SETUP (uncomment if needed)
# ══════════════════════════════════════════════════════════════════════════════
# !pip install pandas numpy vaderSentiment

In [2]:
from __future__ import annotations

import csv
import logging
import re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("highlevel")

print("Setup complete.")

Setup complete.


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
current_dir = Path.cwd()
if (current_dir / "src" / "eda").exists():
    PROJECT_ROOT = current_dir
elif current_dir.name == "eda" and (current_dir.parent.parent / "src").exists():
    PROJECT_ROOT = current_dir.parent.parent
else:
    PROJECT_ROOT = current_dir

INPUT_DIR = PROJECT_ROOT / "src" / "eda" / "input"
OUTPUT_DIR = PROJECT_ROOT / "src" / "eda" / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_ROWS = 20_000          # nrows per dataset for cheap stats (keeps runtime short)
TICKER_TEXT_SAMPLE = 5_000    # rows scanned for ticker diversity
POLARITY_SAMPLE = 2_000       # rows scored for bullish/bearish ratio

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"INPUT_DIR:    {INPUT_DIR}")
print(f"OUTPUT_DIR:   {OUTPUT_DIR}")

PROJECT_ROOT: D:\git\uol-bsc-cm3070-final-project
INPUT_DIR:    D:\git\uol-bsc-cm3070-final-project\src\eda\input
OUTPUT_DIR:   D:\git\uol-bsc-cm3070-final-project\src\eda\output


---
## 1. Discover local CSVs in `src/eda/input/`

Place your manually downloaded + extracted datasets there (one CSV per dataset, or nested folders).

In [4]:
local_csvs = sorted(p for p in INPUT_DIR.rglob("*") if p.suffix.lower() == ".csv")

if not local_csvs:
    print("=" * 60)
    print("No CSVs found in src/eda/input/.")
    print("Download + extract candidate datasets there, then re-run.")
    print("Example: src/eda/input/r_pennystocks_submissions_reddit.csv")
    print("=" * 60)
else:
    print(f"Found {len(local_csvs)} CSV(s):")
    for p in local_csvs:
        print(f"  - {p.relative_to(INPUT_DIR)}  ({p.stat().st_size / 1e6:.1f} MB)")

Found 4 CSV(s):
  - davidwallach_financial-tweets\stockerbot-export.csv  (7.3 MB)
  - leukipp_pennystocks\submissions_reddit.csv  (39.7 MB)
  - leukipp_wallstreetbets\submissions_reddit.csv  (232.4 MB)
  - vivekrathi055_sentiment-analysis-on-financial-tweets\stockerbot-export1.csv  (7.1 MB)


---
## 2. Inline detectors + ticker extraction

In [5]:
def detect_delimiter(filepath: str, sample_size: int = 8192) -> str:
    try:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            sample = f.read(sample_size)
        return csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
    except (csv.Error, OSError):
        return ","


def detect_date_column(df: pd.DataFrame) -> Optional[str]:
    date_keywords = ["date", "timestamp", "created_at", "created_utc", "time", "posted_at", "created"]
    for col in df.columns:
        if col.lower() in date_keywords:
            return col
    for col in df.columns:
        if any(kw in col.lower() for kw in ["date", "time", "created"]):
            return col
    for col in df.select_dtypes(include=["object", "string"]).columns:
        sample = df[col].dropna().head(20)
        if len(sample) == 0:
            continue
        parsed = pd.to_datetime(sample, errors="coerce", format="mixed")
        if parsed.notna().sum() >= len(sample) * 0.8:
            return col
    return None


def detect_text_column(df: pd.DataFrame) -> Optional[str]:
    for kw in ["title", "text", "body", "content", "selftext", "comment", "message"]:
        for col in df.columns:
            if col.lower() == kw:
                return col
    return None


def detect_engagement_columns(df: pd.DataFrame) -> list[str]:
    kw = {"likes", "retweets", "comments", "upvotes", "shares", "favorites", "score",
          "num_comments", "comment_count", "like_count", "retweet_count", "ups", "downs", "comms_num"}
    return [c for c in df.columns if c.lower() in kw]


def detect_sentiment_column(df: pd.DataFrame) -> Optional[str]:
    kw = ["sentiment", "polarity", "sentiment_score", "compound", "bullish", "bearish", "label"]
    for col in df.columns:
        if col.lower() in kw:
            return col
    return None


print("Detectors defined.")

Detectors defined.


In [6]:
KNOWN_TICKERS = {
    "AAPL", "MSFT", "GOOGL", "GOOG", "AMZN", "TSLA", "META", "NVDA", "AMD", "INTC",
    "NFLX", "DIS", "PYPL", "SQ", "SHOP", "ROKU", "GME", "AMC", "BB", "BBBY", "PLTR",
    "WISH", "CLOV", "SOFI", "SPCE", "NIO", "LCID", "RIVN", "HOOD", "DKNG", "COIN",
    "JPM", "BAC", "GS", "MS", "WFC", "C", "V", "MA", "JNJ", "PFE", "MRNA", "BNTX",
    "SPY", "QQQ", "IWM", "ARKK", "SNDL", "BNGO", "SENS", "PROG", "ATOS", "TNXP",
    "HCMC", "CTRM", "MVIS", "TLRY", "WKHS", "CLNE", "RKT", "UWMC", "CRSR",
}
_FALSE_POSITIVES = {
    "CEO", "IPO", "ETF", "SEC", "FDA", "GDP", "CPI", "ATH", "DD", "EPS", "PE", "RSI",
    "OTC", "NYSE", "IMO", "FYI", "YOLO", "FOMO", "HODL", "WSB", "OP", "TLDR", "US",
    "UK", "EU", "AI", "ML", "IT", "PM", "AM", "THE", "AND", "FOR", "ARE", "BUY",
    "PUT", "CALL", "LONG", "SHORT", "BULL", "BEAR", "MOON", "APE", "HOLD", "SELL",
}
_CASHTAG = re.compile(r"\$([A-Z]{2,5})\b")
_UPPER = re.compile(r"\b([A-Z]{2,5})\b")


def extract_tickers(text: str) -> list[str]:
    if not text or not isinstance(text, str):
        return []
    found = set()
    for tag in _CASHTAG.findall(text):
        if tag not in _FALSE_POSITIVES:
            found.add(tag)
    for word in _UPPER.findall(text):
        if word in KNOWN_TICKERS and word not in _FALSE_POSITIVES:
            found.add(word)
    return sorted(found)


print("Ticker extraction defined.")

Ticker extraction defined.


---
## 3. Cheap per-dataset profile

In [7]:
def _bullish_bearish_ratio(series: pd.Series) -> float:
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        analyzer = SentimentIntensityAnalyzer()
    except ImportError:
        return float("nan")
    scores = series.dropna().astype(str).head(POLARITY_SAMPLE).map(
        lambda t: analyzer.polarity_scores(t)["compound"]
    )
    bull = int((scores > 0.05).sum())
    bear = int((scores < -0.05).sum())
    # Cap the ratio to a finite sentinel so the CSV/PDF artifact stays clean
    # (float('inf') serializes poorly). 999.0 means 'all bullish, no bearish'.
    return round(bull / bear, 2) if bear > 0 else (999.0 if bull > 0 else 0.0)


def profile_csv(path: Path, nrows: int = SAMPLE_ROWS) -> dict:
    """Cheap, sampled profile of one CSV."""
    sep = detect_delimiter(str(path))
    # Some candidate CSVs (e.g. stockerbot exports) contain embedded newlines and
    # unescaped quotes inside text fields. The python engine can choke on these
    # ('expected after quote' errors), silently dropping a whole dataset from the
    # comparison. Try tolerant python parsing first, then fall back to the C engine
    # (which handles multiline quoted fields more robustly) before giving up.
    read_kwargs = dict(sep=sep, nrows=nrows, on_bad_lines="skip", encoding_errors="ignore")
    try:
        df = pd.read_csv(path, engine="python", **read_kwargs)
    except (pd.errors.ParserError, csv.Error):
        df = pd.read_csv(path, engine="c", **read_kwargs)

    date_col = detect_date_column(df)
    text_col = detect_text_column(df)
    eng_cols = detect_engagement_columns(df)
    sent_col = detect_sentiment_column(df)

    date_span = "unknown"
    if date_col:
        s = df[date_col]
        if pd.api.types.is_numeric_dtype(s):
            v = s.dropna()
            # Use the median of non-null values (not just row 0, which may be 0/NaN)
            # to decide whether the column holds epoch seconds.
            if len(v) and abs(v.median()) > 1e9:
                dts = pd.to_datetime(s, unit="s", errors="coerce").dropna()
            else:
                dts = pd.to_datetime(s, errors="coerce").dropna()
        else:
            dts = pd.to_datetime(s, errors="coerce").dropna()
        if len(dts):
            date_span = f"{dts.min().date()} → {dts.max().date()}"

    n_tickers = 0
    if text_col:
        uniq: set[str] = set()
        for t in df[text_col].dropna().astype(str).head(TICKER_TEXT_SAMPLE):
            uniq.update(extract_tickers(t))
        n_tickers = len(uniq)

    ratio = _bullish_bearish_ratio(df[text_col]) if text_col else float("nan")

    return {
        # Include the parent folder so datasets with identical filenames
        # (e.g. two 'submissions_reddit.csv') get distinct, unambiguous labels.
        "dataset": f"{path.parent.name}/{path.stem}",
        "path": str(path.relative_to(INPUT_DIR)),
        "sample_rows": len(df),
        "n_columns": df.shape[1],
        "date_col": date_col or "",
        "date_span": date_span,
        "text_col": text_col or "",
        "engagement_cols": ",".join(eng_cols),
        "sentiment_col": sent_col or "",
        "avg_missing_pct": round(float(df.isna().mean().mean()) * 100, 2),
        "unique_tickers_sampled": n_tickers,
        "bullish_bearish_ratio": ratio,
        "has_engagement": bool(eng_cols),
        "has_text": bool(text_col),
        "has_date": bool(date_col),
        "surge_label_ready": bool(eng_cols) and bool(text_col) and bool(date_col),
    }


print("profile_csv defined.")

profile_csv defined.


---
## 4. Profile all datasets

In [8]:
rows: list[dict] = []
for path in local_csvs:
    try:
        rows.append(profile_csv(path))
        print(f"Profiled: {path.name}")
    except Exception as e:
        logger.warning("Failed to profile %s: %s", path.name, e)

print(f"\nProfiled {len(rows)} dataset(s).")

C:\Users\hieun\AppData\Local\Temp\ipykernel_444\1843598572.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dts = pd.to_datetime(s, errors="coerce").dropna()


Profiled: stockerbot-export.csv


Profiled: submissions_reddit.csv


Profiled: submissions_reddit.csv


C:\Users\hieun\AppData\Local\Temp\ipykernel_444\1843598572.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dts = pd.to_datetime(s, errors="coerce").dropna()


Profiled: stockerbot-export1.csv

Profiled 4 dataset(s).


---
## 5. Comparison matrix (presentation only)

This is the deliverable: a side-by-side view for your judgement. `surge_label_ready` flags whether a dataset has the fields needed to build surge labels (text + date + engagement). No dataset is auto-selected; you pick based on this.

The table below is formatted for readability and PDF export.

In [9]:
from IPython.display import display, Markdown

if rows:
    comparison = pd.DataFrame(rows).sort_values(
        ["surge_label_ready", "unique_tickers_sampled", "sample_rows"], ascending=False
    ).reset_index(drop=True)

    # --- Text summary (renders cleanly in PDF) ---
    ready = comparison.loc[comparison["surge_label_ready"], "dataset"].tolist()
    display(Markdown(
        f"**Datasets profiled:** {len(comparison)}  \n"
        f"**Surge-label ready** (has text + date + engagement): "
        f"{', '.join(ready) if ready else 'none'}"
    ))

    # --- PDF-friendly presentation: booleans as Yes/No, tidy headers ---
    PRETTY = {
        "dataset": "Dataset", "path": "Path", "sample_rows": "Rows (sampled)",
        "n_columns": "Columns", "date_col": "Date col", "date_span": "Date span",
        "text_col": "Text col", "engagement_cols": "Engagement cols",
        "sentiment_col": "Sentiment col", "avg_missing_pct": "Missing %",
        "unique_tickers_sampled": "Unique tickers", "bullish_bearish_ratio": "Bull/Bear ratio",
        "has_engagement": "Has engagement", "has_text": "Has text",
        "has_date": "Has date", "surge_label_ready": "Surge-label ready",
    }
    pretty = comparison.copy()
    bool_cols = pretty.select_dtypes(include=["bool"]).columns
    pretty[bool_cols] = pretty[bool_cols].map(lambda b: "Yes" if b else "No")
    pretty = pretty.rename(columns=PRETTY)

    # Transpose: metrics as rows, datasets as columns. Reads well in PDF even with
    # many metrics or long values, and avoids wide tables overflowing the page.
    # (Plain DataFrame display -> HTML table; no jinja2/.style dependency needed.)
    transposed = pretty.set_index("Dataset").T
    transposed.columns.name = "Dataset"
    display(Markdown("**High-level dataset comparison (sampled)**"))
    with pd.option_context("display.max_columns", None, "display.width", None,
                           "display.max_colwidth", None):
        display(transposed)
else:
    comparison = pd.DataFrame()
    display(Markdown("**No datasets profiled.** Add CSVs under `src/eda/input/` (subfolders are scanned) and re-run."))

**Datasets profiled:** 4  
**Surge-label ready** (has text + date + engagement): leukipp_pennystocks/submissions_reddit, leukipp_wallstreetbets/submissions_reddit

**High-level dataset comparison (sampled)**

Dataset,leukipp_pennystocks/submissions_reddit,leukipp_wallstreetbets/submissions_reddit,davidwallach_financial-tweets/stockerbot-export,vivekrathi055_sentiment-analysis-on-financial-tweets/stockerbot-export1
Path,leukipp_pennystocks\submissions_reddit.csv,leukipp_wallstreetbets\submissions_reddit.csv,davidwallach_financial-tweets\stockerbot-export.csv,vivekrathi055_sentiment-analysis-on-financial-tweets\stockerbot-export1.csv
Rows (sampled),20000,20000,20000,20000
Columns,24,24,8,8
Date col,created,created,timestamp,timestamp
Date span,2021-01-01 → 2021-02-16,2021-01-01 → 2021-01-19,2018-02-23 → 2018-07-19,2018-02-23 → 2018-07-19
Text col,title,title,text,text
Engagement cols,"score,num_comments","score,num_comments",,
Sentiment col,,,,
Missing %,0.74,1.53,2.38,2.38
Unique tickers,422,166,2385,2385


---
## 6. Write artifact

In [10]:
out_csv = OUTPUT_DIR / "highlevel_comparison.csv"
if not comparison.empty:
    comparison.to_csv(out_csv, index=False)
    print(f"Wrote comparison matrix: {out_csv}")
else:
    pd.DataFrame(columns=["dataset", "surge_label_ready"]).to_csv(out_csv, index=False)
    print(f"Wrote empty comparison matrix (no data): {out_csv}")

Wrote comparison matrix: D:\git\uol-bsc-cm3070-final-project\src\eda\output\highlevel_comparison.csv


---
## Next step

Pick a dataset by judgement from the matrix above, then open **`03_deep_assessment.ipynb`** and set `TARGET_DATASETS` to the CSV(s) you chose (under `src/eda/input/`).